In [ ]:
# ============================================================
# CELL 6: LOAD LABELED SCORES
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 6 START - LOADING LABELED SCORE FILES")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)
    
    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['model_type']
    topic = CONFIG['workflow']['topic']
    
    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]
    
    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")

# Load the three CSV files using fs.folders
high_path = fs.folders['Cosine_labeling'] / 'scores_high_confidence.csv'
low_path = fs.folders['Cosine_labeling'] / 'scores_low_confidence.csv'
no_path = fs.folders['Cosine_labeling'] / 'scores_no_confidence.csv'

if not high_path.exists() or not low_path.exists() or not no_path.exists():
    print(f"❌ Error: One or more score files not found")
    print(f"   Looking in: {fs.folders['Cosine_labeling']}")
    print(f"   - {high_path.name} {'✓' if high_path.exists() else '✗'}")
    print(f"   - {low_path.name} {'✓' if low_path.exists() else '✗'}")
    print(f"   - {no_path.name} {'✓' if no_path.exists() else '✗'}")
    raise FileNotFoundError("Required score files not found. Please run CHECKPOINT 5 first.")
else:
    high_df = pd.read_csv(high_path)
    low_df = pd.read_csv(low_path)
    no_df = pd.read_csv(no_path)
    
    print(f"✓ Loaded score files from: {fs.folders['Cosine_labeling']}")
    print(f"  High confidence: {len(high_df)} chunks")
    print(f"  Low confidence:  {len(low_df)} chunks")
    print(f"  No confidence:   {len(no_df)} chunks")
    print(f"  Total:           {len(high_df) + len(low_df) + len(no_df)} chunks")

# ============================================================
# CELL 6.1: PREPARE LABELED DATA
# ============================================================

print(f"\n{'='*60}")
print("PREPARING LABELED DATA")
print(f"{'='*60}")

# High confidence = labeled data
df_labeled = high_df.copy()
df_labeled['text'] = df_labeled['raw_text']
df_labeled['label'] = df_labeled['primary_topic']

# Create label mapping
label2id = {label: idx for idx, label in enumerate(sorted(df_labeled['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}
df_labeled['label_id'] = df_labeled['label'].map(label2id)
df_labeled['is_pseudo'] = False

print(f"\nLabel mapping:")
for label, idx in label2id.items():
    count = (df_labeled['label'] == label).sum()
    print(f"  {idx}: {label} ({count} examples)")

print(f"\nTotal labeled examples: {len(df_labeled)}")
# ============================================================
# CELL 6.2: PREPARE PSEUDO-LABELED & UNLABELED DATA
# ============================================================

# =====================
# CONFIGURATION
# =====================
CONFIG = {
    # Sampling limits (balance with labeled data)
    "sampling": {
        "unlabeled_multiplier": 3,  # Max unlabeled = labeled_size * 3
        "pseudo_multiplier": 10      # Max pseudo = labeled_size * 10
    }
}

print(f"\n{'='*60}")
print("PREPARING PSEUDO-LABELED & UNLABELED DATA")
print(f"{'='*60}")

# =====================
# PREPARE PSEUDO-LABELED DATA
# =====================

# Pseudo-labeled pool (low confidence predictions)
df_pseudo = low_df.copy()
df_pseudo['text'] = df_pseudo['raw_text']
df_pseudo['label'] = df_pseudo['primary_topic']
df_pseudo['label_id'] = df_pseudo['label'].map(label2id)
df_pseudo['is_pseudo'] = True

print(f"\nPseudo-labeled pool: {len(df_pseudo)} chunks")

# Sample pseudo-labeled data for balance
max_pseudo = len(df_labeled) * CONFIG["sampling"]["pseudo_multiplier"]
if len(df_pseudo) > max_pseudo:
    df_pseudo_sampled = df_pseudo[['text', 'label', 'label_id', 'is_pseudo']].sample(
        n=max_pseudo, random_state=42
    )
    print(f"  Sampled: {len(df_pseudo_sampled)} (to maintain balance)")
else:
    df_pseudo_sampled = df_pseudo[['text', 'label', 'label_id', 'is_pseudo']].copy()
    print(f"  Using all: {len(df_pseudo_sampled)}")

# =====================
# PREPARE UNLABELED DATA
# =====================

# Unlabeled pool (no confidence predictions)
df_unlabeled = no_df[['raw_text']].copy()
df_unlabeled.rename(columns={'raw_text': 'text'}, inplace=True)
df_unlabeled['label'] = 'UNLABELED'
df_unlabeled['label_id'] = -1
df_unlabeled['is_pseudo'] = False

# Clean: remove empty/null text
df_unlabeled = df_unlabeled[df_unlabeled['text'].notna()].copy()
df_unlabeled = df_unlabeled[df_unlabeled['text'].astype(str).str.strip() != ''].copy()

print(f"\nUnlabeled pool: {len(df_unlabeled)} chunks")

# Sample unlabeled data for balance
max_unlabeled = len(df_labeled) * CONFIG["sampling"]["unlabeled_multiplier"]
if len(df_unlabeled) > max_unlabeled:
    df_unlabeled_sampled = df_unlabeled.sample(n=max_unlabeled, random_state=42)
    print(f"  Sampled: {len(df_unlabeled_sampled)} (to maintain balance)")
else:
    df_unlabeled_sampled = df_unlabeled.copy()
    print(f"  Using all: {len(df_unlabeled_sampled)}")

# =====================
# SUMMARY
# =====================

print(f"\n{'='*60}")
print("DATA PREPARATION SUMMARY")
print(f"{'='*60}")
print(f"  Labeled:     {len(df_labeled)}")
print(f"  Pseudo:      {len(df_pseudo_sampled)}")
print(f"  Unlabeled:   {len(df_unlabeled_sampled)}")
print(f"  Total pool:  {len(df_labeled) + len(df_pseudo_sampled) + len(df_unlabeled_sampled)}")
# ============================================================
# CELL 6.3: CREATE DATASET OPTIONS & TRAIN/VAL SPLIT
# ============================================================

print(f"\n{'='*60}")
print("CREATING DATASET OPTIONS & TRAIN/VAL SPLIT")
print(f"{'='*60}")

# =====================
# STEP 1: GROUP DATA INTO OPTIONS FIRST
# =====================

print(f"\nStep 1: Grouping data into options...")

# =====================
# OPTION 1: LABELED ONLY
# =====================
data_opt1 = df_labeled.copy()

# =====================
# OPTION 2: LABELED + PSEUDO-LABELED
# =====================
data_opt2 = pd.concat([
    df_labeled,
    df_pseudo_sampled
], ignore_index=True)

# =====================
# OPTION 3: LABELED + UNLABELED
# =====================
data_opt3 = pd.concat([
    df_labeled,
    df_unlabeled_sampled
], ignore_index=True)

# =====================
# OPTION 4: ALL (LABELED + PSEUDO + UNLABELED)
# =====================
data_opt4 = pd.concat([
    df_labeled,
    df_pseudo_sampled,
    df_unlabeled_sampled
], ignore_index=True)

print(f"  Option 1 (Labeled only):          {len(data_opt1):>6} examples")
print(f"  Option 2 (Labeled + Pseudo):      {len(data_opt2):>6} examples")
print(f"  Option 3 (Labeled + Unlabeled):   {len(data_opt3):>6} examples")
print(f"  Option 4 (All) ⭐ RECOMMENDED:    {len(data_opt4):>6} examples")

# =====================
# STEP 2: SPLIT EACH OPTION INTO TRAIN/VAL
# =====================

print(f"\nStep 2: Splitting each option into train/val...")

def split_with_stratification(data, option_name):
    """
    Split data into train/val, using stratification if possible.
    Only stratify on labeled data (exclude UNLABELED from stratification).
    """
    # Separate labeled and unlabeled data
    labeled_data = data[data['label'] != 'UNLABELED'].copy()
    unlabeled_data = data[data['label'] == 'UNLABELED'].copy()
    
    # Check if stratified split is possible for labeled data
    if len(labeled_data) > 0:
        topic_counts = labeled_data['label'].value_counts()
        can_stratify = all(topic_counts >= 2)
        
        if can_stratify:
            train_labeled, val_labeled = train_test_split(
                labeled_data,
                test_size=0.2,
                stratify=labeled_data['label'],
                random_state=42
            )
            print(f"  {option_name}: ✓ Stratified split")
        else:
            train_labeled, val_labeled = train_test_split(
                labeled_data,
                test_size=0.2,
                random_state=42
            )
            print(f"  {option_name}: ⚠ Random split (some topics < 2 examples)")
        
        # Add unlabeled data to training set only (not validation)
        if len(unlabeled_data) > 0:
            train_data = pd.concat([train_labeled, unlabeled_data], ignore_index=True)
            print(f"      Added {len(unlabeled_data)} unlabeled to training")
        else:
            train_data = train_labeled
        
        val_data = val_labeled
    else:
        # Edge case: only unlabeled data (shouldn't happen but handle it)
        train_data = data
        val_data = data.head(0)  # Empty validation set
        print(f"  {option_name}: ⚠ No labeled data for validation")
    
    return train_data, val_data

# Split each option
train_opt1, val_opt1 = split_with_stratification(data_opt1, "Option 1")
train_opt2, val_opt2 = split_with_stratification(data_opt2, "Option 2")
train_opt3, val_opt3 = split_with_stratification(data_opt3, "Option 3")
train_opt4, val_opt4 = split_with_stratification(data_opt4, "Option 4")

# =====================# DETAILED TOPIC BREAKDOWN TABLE# =====================print(f"\n{'='*80}")print("TRAIN/VAL SPLIT SUMMARY - PER TOPIC BREAKDOWN")print(f"{'='*80}")# Helper function to classify confidence tiersdef classify_confidence(row):    """Classify chunk into high/low/no confidence based on original tier"""    if 'confidence_level' in row:        return row['confidence_level']    # Fallback: use label_id to infer    if row.get('label_id', -1) == -1:        return 'none'    elif row.get('is_pseudo', False):        return 'low'    else:        return 'high'# Get all unique topics (excluding UNLABELED)all_topics = sorted([t for t in label2id.keys() if t != 'UNLABELED'])# Build comprehensive table for chosen dataset optiondataset_option = CONFIG.get('training', {}).get('dataset_option', 'option4')if dataset_option == 'option1':    train_data, val_data = train_opt1, val_opt1elif dataset_option == 'option2':    train_data, val_data = train_opt2, val_opt2elif dataset_option == 'option3':    train_data, val_data = train_opt3, val_opt3else:    train_data, val_data = train_opt4, val_opt4print(f"\nDataset Option: {dataset_option}\n")# Create table headerheader = f"{'Topic':<30} | {'Train Split':<45} | {'Val Split':<45} | Total"separator = "-" * len(header)print(separator)print(header)print(separator)# Track totalstotal_train_high, total_train_low, total_train_none = 0, 0, 0total_val_high, total_val_low, total_val_none = 0, 0, 0# Process each topictopic_stats = []for topic in all_topics:    # Train split    train_topic = train_data[train_data['label'] == topic]    # Count by confidence level in training    train_high = len(train_topic[(train_topic.get('is_pseudo', False) == False) &                                   (train_topic['label_id'] != -1)])    train_low = len(train_topic[train_topic.get('is_pseudo', False) == True])    train_none = len(train_topic[train_topic['label_id'] == -1])    # Val split    val_topic = val_data[val_data['label'] == topic]    # Count by confidence level in validation    val_high = len(val_topic[(val_topic.get('is_pseudo', False) == False) &                              (val_topic['label_id'] != -1)])    val_low = len(val_topic[val_topic.get('is_pseudo', False) == True])    val_none = len(val_topic[val_topic['label_id'] == -1])    topic_total = train_high + train_low + train_none + val_high + val_low + val_none    # Format row    train_str = f"H:{train_high:>4} L:{train_low:>4} N:{train_none:>4}"    val_str = f"H:{val_high:>4} L:{val_low:>4} N:{val_none:>4}"    print(f"{topic:<30} | {train_str:<45} | {val_str:<45} | {topic_total:>5}")    # Accumulate totals    total_train_high += train_high    total_train_low += train_low    total_train_none += train_none    total_val_high += val_high    total_val_low += val_low    total_val_none += val_none    # Store for diagnostics    topic_stats.append({        'topic': topic,        'train_high': train_high,        'train_low': train_low,        'train_none': train_none,        'val_high': val_high,        'val_low': val_low,        'val_none': val_none,        'total': topic_total    })print(separator)# Grand totalstotal_train_str = f"H:{total_train_high:>4} L:{total_train_low:>4} N:{total_train_none:>4}"total_val_str = f"H:{total_val_high:>4} L:{total_val_low:>4} N:{total_val_none:>4}"grand_total = total_train_high + total_train_low + total_train_none + total_val_high + total_val_low + total_val_noneprint(f"{'TOTAL':<30} | {total_train_str:<45} | {total_val_str:<45} | {grand_total:>5}")print(separator)print(f"\nLegend: H=High confidence, L=Low confidence (pseudo), N=No confidence (unlabeled)")# =====================# PER-TOPIC DIAGNOSTICS# =====================print(f"\n{'='*80}")print("PER-TOPIC DIAGNOSTICS")print(f"{'='*80}")import numpy as np# Calculate imbalance metricscounts = [s['train_high'] for s in topic_stats]if min(counts) > 0:    imbalance_ratio = max(counts) / min(counts)else:    imbalance_ratio = float('inf')print(f"\n1. CLASS IMBALANCE (High-confidence training data):")print(f"   Imbalance ratio: {imbalance_ratio:.2f}x (max/min)")if imbalance_ratio > 3:    print(f"   ⚠️  SIGNIFICANT IMBALANCE - Consider topic weighting in training")elif imbalance_ratio > 2:    print(f"   ⚠️  Moderate imbalance - Monitor per-topic F1 scores")else:    print(f"   ✓ Balanced training data")# Calculate cosine score statistics per topicprint(f"\n2. COSINE SCORE STATISTICS (from source data):")# We need to look at the original high/low/no dataframesfor topic in all_topics:    high_topic = high_df[high_df['primary_topic'] == topic]    low_topic = low_df[low_df['primary_topic'] == topic]    if len(high_topic) > 0:        high_mean = high_topic['max_score'].mean()        high_std = high_topic['max_score'].std()        high_min = high_topic['max_score'].min()        high_max = high_topic['max_score'].max()        print(f"\n   {topic}:")        print(f"     High-conf: n={len(high_topic):>4}, cos_mean={high_mean:.3f}, std={high_std:.3f}, range=[{high_min:.3f}, {high_max:.3f}]")        if len(low_topic) > 0:            low_mean = low_topic['max_score'].mean()            low_std = low_topic['max_score'].std()            print(f"     Low-conf:  n={len(low_topic):>4}, cos_mean={low_mean:.3f}, std={low_std:.3f}")        # Quality indicator        if high_mean < 0.55:            print(f"     ⚠️  Low average cosine - weak dictionary match for this topic")        elif high_mean > 0.70:            print(f"     ✓ Strong dictionary match")# Recommendationprint(f"\n3. RECOMMENDATIONS:")print(f"   Based on the diagnostics above:")if imbalance_ratio > 3:    print(f"   - Enable topic weighting: CONFIG['training']['use_topic_weights'] = True")    print(f"   - Use strategy: 'sqrt_inverse_freq' for softer weighting")# Check if any topic has very low high-confidence countsmin_high = min([s['train_high'] for s in topic_stats])if min_high < 50:    print(f"   - Some topics have <50 high-confidence examples")    print(f"   - Consider including pseudo-labels (option2 or option4)")# Check cosine qualityall_means = []for topic in all_topics:    high_topic = high_df[high_df['primary_topic'] == topic]    if len(high_topic) > 0:        all_means.append(high_topic['max_score'].mean())if all_means and max(all_means) - min(all_means) > 0.15:    print(f"   - Cosine score variance >0.15 across topics")    print(f"   - Some topics may have weaker dictionary coverage")    print(f"   - Consider expanding dictionary for low-scoring topics")print(f"\n{'='*80}")# SAVE DATA
# =====================

print(f"\n{'='*60}")
print("SAVING DATA")
print(f"{'='*60}")

# Save label mapping
fs.save_data(label2id, "bertje_label_mapping", "Model_finetuning", "json")
print(f"✓ Saved label mapping")

# Save all training and validation sets
fs.save_data(train_opt1, "train_data_option1_labeled_only", "Model_finetuning", "csv")
fs.save_data(val_opt1, "val_data_option1", "Model_finetuning", "csv")

fs.save_data(train_opt2, "train_data_option2_with_pseudo", "Model_finetuning", "csv")
fs.save_data(val_opt2, "val_data_option2", "Model_finetuning", "csv")

fs.save_data(train_opt3, "train_data_option3_with_unlabeled", "Model_finetuning", "csv")
fs.save_data(val_opt3, "val_data_option3", "Model_finetuning", "csv")

fs.save_data(train_opt4, "train_data_option4_all", "Model_finetuning", "csv")
fs.save_data(val_opt4, "val_data_option4", "Model_finetuning", "csv")

print(f"✓ Saved all training and validation sets")

# Save checkpoint
fs.save_config("checkpoint6_training_prep")
print(f"✓ Checkpoint saved")

print(f"\n{'='*60}")
print("✓ DATA PREPARATION COMPLETE")
print(f"{'='*60}")
print(f"\nReady for model training!")
print(f"Choose one of the training options (1-4) for your model.")

In [ ]:
# ============================================================
# CELL 7.1: SETUP TRAINING
# ============================================================

try:
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        TrainingArguments,
        Trainer,
        DataCollatorWithPadding
    )
    from datasets import Dataset
    import torch
    
    print("✓ Transformers library available")
    
    # Check GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
    
    TRAINING_AVAILABLE = True
    
except ImportError as e:
    print("⚠ Transformers library not available")
    print("  Install: pip install transformers datasets torch")
    TRAINING_AVAILABLE = False
# ============================================================
# CELL 7.2: LOAD & PREPARE DATA  (adds cos_* columns for soft labels)
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("PREPARING TRAINING DATA")
    print(f"{'='*60}")
    
    # -----------------------------[ 1) PICK DATASET ]-----------------------------
    dataset_option = CONFIG['training']['dataset_option']
    if dataset_option == 'option1':
        train_dataset = train_opt1
        val_dataset   = val_opt1
    elif dataset_option == 'option2':
        train_dataset = train_opt2
        val_dataset   = val_opt2
    elif dataset_option == 'option3':
        train_dataset = train_opt3
        val_dataset   = val_opt3
    else:
        train_dataset = train_opt4
        val_dataset   = val_opt4
    
    print(f"\nUsing {dataset_option}: {len(train_dataset)} examples")
    
    # -----------------------------[ 2) LOAD MODEL + TOKENIZER ]-------------------
    from pathlib import Path
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
    
    if CONFIG['model']['use_pretrained'] and CONFIG['paths']['pretrained_model_path']:
        model_path = CONFIG['paths']['pretrained_model_path']
        if Path(model_path).exists():
            model_name = model_path
            print(f"✓ Loading pretrained model from: {model_path}")
        else:
            model_name = "GroNLP/bert-base-dutch-cased"
            print(f"⚠ Pretrained path not found, using base model")
    else:
        model_name = "GroNLP/bert-base-dutch-cased"
        print(f"✓ Using base model: GroNLP/bert-base-dutch-cased")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id
    )
    model.to(device)
    print(f"\n✓ Model loaded: {model_name}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # -----------------------------[ 3) TOKENIZER FN ]-----------------------------
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            padding=False,
            truncation=True,
            max_length=512
        )
    
    # -----------------------------[ 4) PICK COSINE COLUMNS ]----------------------
    # Auto-detect columns that start with 'cos_' or use CONFIG['cosine']['columns']
    import numpy as np
    from datasets import Dataset
    
    train_cols = list(train_dataset.columns)
    default_cos_cols = [c for c in train_cols if isinstance(c, str) and c.startswith("cos_")]
    cos_cfg = CONFIG.get('cosine', {}) if isinstance(CONFIG.get('cosine', {}), dict) else {}
    cos_cols = cos_cfg.get('columns', default_cos_cols)

    # CRITICAL: Sort cos_cols to match label2id ordering
    # label2id was created as: {label: idx for idx, label in enumerate(sorted(...))}
    # So we must sort cos_cols by their topic names to match
    if cos_cols and 'id2label' in globals():
        # Build ordered list from id2label (which matches label2id)
        sorted_topics = [id2label[i] for i in sorted(id2label.keys())]
        # Match cos_* columns to topics - handle underscores and spaces
        cos_cols_ordered = []
        for topic in sorted_topics:
            # Try exact match with 'cos_' prefix
            for col in cos_cols:
                col_topic = col.replace('cos_', '').replace('_', ' ')
                if col_topic == topic or col_topic.replace(' ', '_') == topic.replace(' ', '_'):
                    cos_cols_ordered.append(col)
                    break
        if len(cos_cols_ordered) == len(cos_cols):
            cos_cols = cos_cols_ordered
            print(f"  ✓ Sorted cos_cols to match label2id order")
        else:
            print(f"  ⚠ Warning: Could not match all cos_cols to topics. Using original order.")
    
    # Keep only numeric cosine columns that exist in both splits
    cos_cols = [c for c in cos_cols if c in train_dataset.columns and c in val_dataset.columns]
    # Optional: ensure they are numeric
    for c in list(cos_cols):
        try:
            _ = np.asarray(train_dataset[c].astype(float))
            _ = np.asarray(val_dataset[c].astype(float))
        except Exception:
            print(f"  ⚠ Skipping non-numeric cosine column: {c}")
            cos_cols.remove(c)
    
    if len(cos_cols) == 0:
        print("ℹ No cos_* columns found. Training will fall back to hard labels.")
    else:
        print(f"✓ Cosine columns detected for soft labels: {cos_cols}")
    
    # -----------------------------[ 5) BUILD HF DATASETS ]------------------------
    # Include labels and all detected cos_* columns alongside tokenized inputs
    # Note: do not drop cos_* when mapping or set_format is applied
    base_train_cols = ['text', 'label_id'] + cos_cols
    base_val_cols   = ['text', 'label_id'] + cos_cols
    
    # guard against missing columns due to prior filtering
    base_train_cols = [c for c in base_train_cols if c in train_dataset.columns]
    base_val_cols   = [c for c in base_val_cols   if c in val_dataset.columns]
    
    train_labeled = train_dataset[train_dataset['label_id'] != -1].copy()
    hf_train = Dataset.from_pandas(train_labeled[base_train_cols].reset_index(drop=True))
    hf_val   = Dataset.from_pandas(val_dataset[base_val_cols].reset_index(drop=True))
    
    # map tokenizer (do not remove cosine columns)
    hf_train = hf_train.map(tokenize_function, batched=True, remove_columns=['text'])
    hf_val   = hf_val.map(  tokenize_function, batched=True, remove_columns=['text'])
    
    # rename label column
    hf_train = hf_train.rename_column('label_id', 'labels')
    hf_val   = hf_val.rename_column('label_id', 'labels')
    
    # set tensor format, include cos_* so collator can see them
    tensor_cols_train = ['input_ids', 'attention_mask', 'labels'] + cos_cols
    tensor_cols_val   = ['input_ids', 'attention_mask', 'labels'] + cos_cols
    hf_train.set_format(type='torch', columns=tensor_cols_train)
    hf_val.set_format(type='torch',   columns=tensor_cols_val)
    
    # data collator stays the same here, CELL 7.3 will wrap it when needed
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    print(f"\n✓ Data prepared")
    print(f"  Train: {len(hf_train)}, Val: {len(hf_val)}")
    if len(cos_cols) > 0:
        print(f"  Included cosine columns in HF datasets: {cos_cols}")
    else:
        print(f"  No cosine columns included (none detected)")

# ============================================================
# CELL 7.3: TRAIN MODEL  (BERTje with cosine soft labels + reject gate)
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("TRAINING MODEL  (BERTje with cosine soft labels + reject gate)")
    print(f"{'='*60}")

    # -----------------------------[ IMPORTS ]-----------------------------
    import os, json
    import numpy as np
    import torch
    import torch.nn.functional as F
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    from transformers import TrainingArguments, Trainer
    from transformers import DataCollatorWithPadding

    # -----------------------------[ 0) METRICS: unchanged ]----------------
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted', zero_division=0
        )
        return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    # -----------------------------[ 1) COSINE SOFT-LABEL SETUP ]----------
    print("\n[SETUP] Detecting cosine columns and soft-label settings...")
    num_labels = int(getattr(model.config, "num_labels", 3))

    # Note: cos_cols already detected and sorted in CELL 7.2
    cos_tau = float(cos_cfg.get('softmax_tau', 0.5))         # temperature for soft labels from cosine
    other_threshold = float(cos_cfg.get('other_threshold', 0.40))  # low max-cos => likely irrelevant
    other_weight = float(cos_cfg.get('other_weight', 0.30))        # weight floor for likely-irrelevant

    # -----------------------------[ 2) DATA COLLATOR (passes cos_*) ]------
    if cos_cols:
        base_collator = data_collator if data_collator is not None else DataCollatorWithPadding(tokenizer)

        class CollatorWithCosine:
            def __init__(self, base, cos_columns):
                self.base = base
                self.cos_columns = list(cos_columns)
            def __call__(self, features):
                # keep cosine values before base collator filters anything
                cos_buf = {c: [float(f.get(c, 0.0)) for f in features] for c in self.cos_columns}
                batch = self.base(features)  # turns text parts into tensors
                # add cosine tensors
                for c, vals in cos_buf.items():
                    batch[c] = torch.tensor(vals, dtype=torch.float)
                return batch

        effective_collator = CollatorWithCosine(base_collator, cos_cols)
        print(f"  ✓ Using soft labels from cosine columns: {cos_cols}")
        print(f"  ✓ Cosine softmax tau: {cos_tau}, irrelevant gate during training (thr={other_threshold}, floor={other_weight})")
    else:
        effective_collator = data_collator
        print("  ✓ Proceeding with hard-label training (no cosine columns used)")

    # -----------------------------[ 3) CUSTOM TRAINER (fix: accept num_items_in_batch) ]---
    class SoftLabelTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.get("labels")
            # strip out non-model keys
            model_inputs = {k: v for k, v in inputs.items() if k not in (["labels"] + list(cos_cols))}
            outputs = model(**model_inputs)  # standard forward
            logits = outputs.logits  # [B, C]

            if cos_cols:
                cos_stack = torch.stack([inputs[c].float() for c in cos_cols], dim=1)  # [B, C]
                # temperature softmax on cosine scores -> soft targets
                cos_norm = (cos_stack / max(cos_tau, 1e-6)).softmax(dim=1)            # [B, C]
                # per-example weights (downweight likely-irrelevant)
                w = cos_stack.max(dim=1).values
                w = torch.where(w < other_threshold, torch.full_like(w, other_weight), w)
                w = torch.clamp(w, min=1e-3)
                # cross-entropy with soft labels
                logp = F.log_softmax(logits, dim=1)                                   # [B, C]
                loss_vec = -(cos_norm * logp).sum(dim=1)                               # [B]
                loss = (loss_vec * w).sum() / w.sum()
            else:
                loss = F.cross_entropy(logits, labels)

            return (loss, outputs) if return_outputs else loss

    # -----------------------------[ 4) TRAINING ARGS: add remove_unused_columns=False ]----
    model_output_dir = str(fs.folders['Model_finetuning'])

    training_args = TrainingArguments(
        output_dir=model_output_dir,
        num_train_epochs=CONFIG['training']['num_epochs'],
        per_device_train_batch_size=CONFIG['training']['batch_size_train'],
        per_device_eval_batch_size=CONFIG['training']['batch_size_eval'],
        learning_rate=CONFIG['training']['learning_rate'],
        weight_decay=CONFIG['training']['weight_decay'],
        warmup_ratio=CONFIG['training']['warmup_ratio'],
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=str(fs.folders['Model_finetuning'] / "logs"),
        logging_strategy="steps",
        logging_steps=50,
        report_to=None,
        fp16=torch.cuda.is_available(),
        seed=42,
        push_to_hub=False,
        remove_unused_columns=False,  # <<< IMPORTANT so cos_* reach the collator
    )

    # -----------------------------[ 5) TRAIN ]-----------------------------
    trainer = SoftLabelTrainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        tokenizer=tokenizer,
        data_collator=effective_collator,
        compute_metrics=compute_metrics,
    )

    print(f"\nStarting training for {CONFIG['training']['num_epochs']} epochs...")
    train_result = trainer.train()

    print(f"\n{'='*60}")
    print("TRAINING COMPLETE")
    print(f"{'='*60}")

    # -----------------------------[ 6) SAVE MODEL + TOKENIZER: unchanged ]-
    trainer.save_model(model_output_dir)
    tokenizer.save_pretrained(model_output_dir)

    # -----------------------------[ 7) EVALUATE: unchanged ]---------------
    eval_results = trainer.evaluate()

    print(f"\nValidation Results:")
    print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
    print(f"  Precision: {eval_results['eval_precision']:.4f}")
    print(f"  Recall:    {eval_results['eval_recall']:.4f}")
    print(f"  F1 Score:  {eval_results['eval_f1']:.4f}")

    # -----------------------------[ 8) NEW: save soft-label info ]---------
    soft_info = {
        "used_cosine_columns": cos_cols,
        "cosine_softmax_tau": cos_tau,
        "other_threshold": other_threshold,
        "other_weight_floor": other_weight,
        "num_labels": int(num_labels),
        "used_soft_labels": bool(cos_cols)
    }
    fs.save_data(soft_info, "softlabel_training_info", "Model_finetuning", "json")
    print("✓ Saved: Model_finetuning/softlabel_training_info.json")

    # -----------------------------[ 9) NEW: calibrate reject tiers ]-------
    print("\n[CALIBRATION] Fitting temperature and computing tier thresholds...")
    val_pred = trainer.predict(hf_val)
    val_logits = val_pred.predictions
    val_labels = np.array(hf_val["labels"])

    def nll_with_T(T):
        T = max(0.05, float(T))
        z = torch.tensor(val_logits) / T
        logp = torch.log_softmax(z, dim=1).numpy()
        return -float(np.mean([logp[i, val_labels[i]] for i in range(len(val_labels))]))

    T = 1.0
    for _ in range(20):  # cheap line search
        cands = [max(0.05, T * f) for f in (0.5, 0.75, 1.0, 1.25, 1.5)]
        losses = [nll_with_T(c) for c in cands]
        T = cands[int(np.argmin(losses))]

    p_cal = torch.softmax(torch.tensor(val_logits) / T, dim=1).numpy()
    pmax_cal = p_cal.max(axis=1)

    # Target shares for tiers (simple defaults; override in CONFIG['cosine'] if you want)
    hi_share  = float(cos_cfg.get('target_high_share', 0.30))
    med_share = float(cos_cfg.get('target_med_share', 0.30))

    hi_tau  = float(np.quantile(pmax_cal, 1.0 - min(max(hi_share, 0.01), 0.95)))
    med_tau = float(np.quantile(pmax_cal, 1.0 - min(max(hi_share + med_share, 0.02), 0.98)))

    calib = {
        "temperature_T": T,
        "tier_thresholds": {
            "HIGH":   hi_tau,
            "MEDIUM": med_tau,
            "LOW_or_IRRELEVANT": 0.0
        },
        "note": "At inference: softmax(logits / T). If pmax≥HIGH→HIGH, if MEDIUM≤pmax<HIGH→MEDIUM, else LOW/Irrelevant."
    }
    fs.save_data(calib, "bert_temperature_and_tiers", "Model_finetuning", "json")
    print("✓ Saved: Model_finetuning/bert_temperature_and_tiers.json")

    # -----------------------------[ 10) SAVE METRICS + CHECKPOINT: same ]--
    metrics = {
        "train_loss": train_result.training_loss,
        "train_runtime": train_result.metrics['train_runtime'],
        "eval_accuracy": eval_results['eval_accuracy'],
        "eval_precision": eval_results['eval_precision'],
        "eval_recall": eval_results['eval_recall'],
        "eval_f1": eval_results['eval_f1'],
        "eval_loss": eval_results['eval_loss'],
        "num_train_examples": len(hf_train),
        "num_eval_examples": len(hf_val),
        "num_epochs": CONFIG['training']['num_epochs'],
        "dataset_used": dataset_option,
    }
    fs.save_data(metrics, "training_metrics", "Model_finetuning", "json")
    fs.save_config("checkpoint7_trained")

    print(f"\n✓ Model and metrics saved")
    print("✓ Soft labels used" if cos_cols else "✓ Hard labels used (no cosine columns detected)")

else:
    print("⚠ Skipping training - transformers library not available")
